# CS224N L01 Code Capsule: One-hot vs Dense Vectors

**Waypoint**: WP01 — 为什么需要词向量？

**Official anchor**: Slides p19-22; Notes §2.2 (one-hot vectors, Eq.1-2)

这个 notebook 演示为什么 one-hot 编码无法表示词义相似性，以及密集向量如何解决这个问题。

## 核心问题

搜索 "Seattle motel" 应该匹配 "Seattle hotel"，但 one-hot 向量下这两个词完全正交（点积=0），
模型无法区分 "hotel≈motel" 和 "book≠fish"。

## 你将看到

1. One-hot 编码：每个词一个位置为 1，其余为 0
2. One-hot 点积：所有不同词对的点积都是 0
3. 密集向量：低维连续向量，相似词在向量空间中距离近
4. Cosine 相似度：密集向量能区分相似和不相似的词对

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Define a small vocabulary
vocab = ["hotel", "motel", "book", "cat", "dog", "fish"]
vocab_size = len(vocab)
print(f"Vocabulary ({vocab_size} words): {vocab}")

## One-hot Encoding

每个词用一个 $|V|$ 维向量表示，只有自己的位置是 1，其余全是 0。

对应 Slides p19：`motel = [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]`

In [ ]:
# 2. Build one-hot vectors
one_hot = {}
for i, word in enumerate(vocab):
    vec = [0] * vocab_size
    vec[i] = 1
    one_hot[word] = vec

for word, vec in one_hot.items():
    print(f"  {word:>6s} = {vec}")

print(f"\nDimension = vocab_size = {vocab_size}")

## The Fatal Problem: All Dot Products = 0

对应 Slides p20："There is no natural notion of similarity for one-hot vectors!"

任意两个不同的 one-hot 向量点积都是 0（正交），模型无法区分近义词和无关词。

In [ ]:
# 3. Dot products -- all zero!
pairs = [
    ("hotel", "motel", "synonyms -- should be similar"),
    ("cat", "dog", "both animals -- should be somewhat similar"),
    ("book", "fish", "unrelated -- should be dissimilar"),
]

print("--- Dot Products (One-hot) ---")
for w1, w2, desc in pairs:
    dot = float(np.dot(one_hot[w1], one_hot[w2]))
    print(f"  {w1} . {w2} = {dot:.1f}   ({desc})")

print("\nALL dot products are 0.0 -- one-hot vectors are orthogonal!")

## Dense Vectors: Learning to Encode Similarity

对应 Slides p22：dense vectors where similar words have similar vectors.

下面是手工设定的 2D 密集向量（实际 word2vec 从数据中学习，维度通常 100-300）。
我们把语义相近的词放在向量空间中相近的位置。

In [ ]:
# 4. Dense vectors (2D illustration)
dense_2d = {
    "hotel": [0.8, 0.7],
    "motel": [0.7, 0.75],
    "book":  [-0.5, 0.3],
    "cat":   [0.1, -0.8],
    "dog":   [0.3, -0.6],
    "fish":  [-0.2, -0.9],
}

for word, vec in dense_2d.items():
    print(f"  {word:>6s} = [{vec[0]:+.2f}, {vec[1]:+.2f}]")

In [ ]:
# 5. Cosine similarity on dense vectors
def cosine_similarity(v1, v2):
    v1, v2 = np.array(v1), np.array(v2)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

print("--- Cosine Similarity (Dense Vectors) ---")
for w1, w2, desc in pairs:
    sim = cosine_similarity(dense_2d[w1], dense_2d[w2])
    print(f"  cos({w1}, {w2}) = {sim:.4f}   ({desc})")

print("\nDense vectors encode similarity!")

## Visualization

左图：One-hot 矩阵（单位矩阵）——所有行互相正交。
右图：密集向量在 2D 空间中的分布——相似词聚在一起。

In [ ]:
# 6. Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: One-hot heatmap
ax = axes[0]
oh_matrix = np.array([one_hot[w] for w in vocab])
im = ax.imshow(oh_matrix, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(vocab_size))
ax.set_xticklabels(vocab, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(vocab_size))
ax.set_yticklabels(vocab, fontsize=9)
ax.set_title("One-hot Vectors (identity matrix)\nAll pairs orthogonal: dot = 0", fontsize=11)
for i in range(vocab_size):
    for j in range(vocab_size):
        val = oh_matrix[i, j]
        color = "white" if val > 0.5 else "black"
        ax.text(j, i, str(int(val)), ha="center", va="center", fontsize=9, color=color)
plt.colorbar(im, ax=ax, shrink=0.8)

# Right: Dense vectors in 2D
ax = axes[1]
colors_map = {
    "hotel": "#e74c3c", "motel": "#e67e22",
    "book": "#3498db",
    "cat": "#2ecc71", "dog": "#27ae60", "fish": "#1abc9c",
}
for word, vec in dense_2d.items():
    ax.scatter(vec[0], vec[1], s=200, c=colors_map[word], zorder=5,
               edgecolors='black', linewidth=0.5)
    ax.annotate(word, (vec[0], vec[1]), fontsize=10, fontweight='bold',
                xytext=(5, 5), textcoords='offset points')

# Draw similarity lines
for w1, w2, color in [("hotel","motel","#e74c3c"), ("cat","dog","#27ae60")]:
    v1, v2 = dense_2d[w1], dense_2d[w2]
    ax.plot([v1[0], v2[0]], [v1[1], v2[1]], '--', color=color, alpha=0.7, linewidth=1.5)
    sim = cosine_similarity(v1, v2)
    mid = ((v1[0]+v2[0])/2, (v1[1]+v2[1])/2)
    ax.text(mid[0], mid[1]+0.05, f"cos={sim:.3f}", fontsize=8, ha='center',
            color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                      edgecolor=color, alpha=0.8))

ax.set_title("Dense Vectors (2D)\nSimilar words are close together", fontsize=11)
ax.set_xlabel("Dimension 1")
ax.set_ylabel("Dimension 2")
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.show()

## 总结

| 对比 | One-hot | Dense (learned) |
|------|---------|----------------|
| 维度 | = 词汇量 (50,000+) | 通常 100-300 |
| 稀疏性 | 极度稀疏（只有一个 1） | 密集（每维都有值） |
| 相似性编码 | ❌ 所有不同词对点积=0 | ✅ cosine 反映语义关系 |

这就是 CS224N L01 的核心动机：**从 one-hot 到从数据中学习的密集向量**。